### importing libraries

In [19]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

### importing data

This code loads the simplified version of Google's GoEmotions dataset directly from Hugging Face using pandas. It defines paths to three parquet files (train, validation, test) and loads the training set into a dataframe. The dataset contains around 43,000 English Reddit comments labeled with 27 emotions plus neutral. The simplified version has binary labels for each emotion rather than multi-label annotations. we will deal with this later since we want to keep the labeling consistent for the test data and training data

In [25]:
import pandas as pd

splits = {'train': 'simplified/train-00000-of-00001.parquet', 'validation': 'simplified/validation-00000-of-00001.parquet', 'test': 'simplified/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["train"])

here we map the binary labels to the right core emotion. We do this so the labels are consistent between the training and test data. This way we can accurately asses the accuracy of our model.

In [26]:
import pandas as pd


# Map numbers to our 7 emotions
emotion_mapping = {
    0: 'happiness',  # admiration
    1: 'happiness',  # amusement
    2: 'anger',      # anger
    3: 'anger',      # annoyance
    4: 'happiness',  # approval
    5: 'happiness',  # caring
    6: 'surprise',   # confusion
    7: 'neutral',    # curiosity
    8: 'neutral',    # desire
    9: 'sadness',    # disappointment
    10: 'anger',     # disapproval
    11: 'disgust',   # disgust
    12: 'sadness',   # embarrassment
    13: 'happiness', # excitement
    14: 'fear',      # fear
    15: 'happiness', # gratitude
    16: 'sadness',   # grief
    17: 'happiness', # joy
    18: 'happiness', # love
    19: 'fear',      # nervousness
    20: 'happiness', # optimism
    21: 'happiness', # pride
    22: 'surprise',  # realization
    23: 'happiness', # relief
    24: 'sadness',   # remorse
    25: 'sadness',   # sadness
    26: 'surprise',  # surprise
    27: 'neutral'    # neutral
}

# Map the labels column
def map_labels(labels_list):
    if len(labels_list) == 0:
        return 'neutral'
    return emotion_mapping[labels_list[0]]

df['emotion'] = df['labels'].apply(map_labels)

# Keep only text and emotion
df = df[['text', 'emotion']]

print(df['emotion'].value_counts())

emotion
happiness    16405
neutral      15138
anger         5336
surprise      2717
sadness       2619
fear           615
disgust        580
Name: count, dtype: int64


We would like to use all the data as training data since we already have test data available from the client pipeline

In [27]:
df_train = df

here we import the test data and we test on the manually labeled data named 'corrected'

In [33]:
df_test = pd.read_excel("test_improved.xlsx")
df_test = df_test.sample(n=len(df_test))

I first checked if CUDA was available and verified my GPU was being detected properly. I loaded the SamLowe RoBERTa model that was pre-trained on GoEmotions directly using AutoModelForSequenceClassification instead of the pipeline to have better control over GPU usage. I explicitly moved the model to CUDA to ensure it would run on my RTX 4070.
Since the RoBERTa-GoEmotions model predicts 28 different emotions and I only needed 7 categories for my task, I created a mapping dictionary to consolidate the predictions. I mapped emotions like joy, love, and gratitude all to happiness, while emotions like grief and remorse went to sadness. I made curiosity map to surprise which might be debatable but seemed reasonable for my use case.

I wrote a batch prediction function that processes texts in chunks of 32 (later increased to 64) to efficiently use GPU memory. I made sure to put the model in eval mode and used torch.no_grad() to prevent gradient computation during inference. I tokenized each batch with padding and truncation, moved the tensors to GPU, and then converted the model's 28-class predictions to my 7 emotion categories.

I tested the model on my test set and timed the predictions to verify GPU acceleration was working - it processed everything in under a second. I calculated all the standard metrics using sklearn and printed a detailed classification report to see per-class performance. I used the 'corrected' column from my dataframe as ground truth since I had manually fixed some labeling issues in the original test set.

In [34]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load model directly (not pipeline) for GPU control
model_name = "SamLowe/roberta-base-go_emotions"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Force GPU
model = model.cuda()
print(f"Model on GPU: {next(model.parameters()).is_cuda}")

# GoEmotions has 28 emotions - map to your 7
goemotions_to_7 = {
    'admiration': 'happiness',
    'amusement': 'happiness',
    'anger': 'anger',
    'annoyance': 'anger',
    'approval': 'happiness',
    'caring': 'happiness',
    'confusion': 'surprise',
    'curiosity': 'surprise',
    'desire': 'neutral',
    'disappointment': 'sadness',
    'disapproval': 'anger',
    'disgust': 'disgust',
    'embarrassment': 'sadness',
    'excitement': 'happiness',
    'fear': 'fear',
    'gratitude': 'happiness',
    'grief': 'sadness',
    'joy': 'happiness',
    'love': 'happiness',
    'nervousness': 'fear',
    'optimism': 'happiness',
    'pride': 'happiness',
    'realization': 'surprise',
    'relief': 'happiness',
    'remorse': 'sadness',
    'sadness': 'sadness',
    'surprise': 'surprise',
    'neutral': 'neutral'
}

# GoEmotions label order
go_emotions_labels = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
                      'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
                      'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude',
                      'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride',
                      'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

# Predict function
def predict_emotions_gpu(texts, batch_size=32):
    all_predictions = []
    model.eval()
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize and move to GPU
        inputs = tokenizer(batch_texts, padding=True, truncation=True, 
                          max_length=512, return_tensors='pt')
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # Predict on GPU
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_classes = torch.argmax(predictions, dim=-1)
        
        # Convert to your 7 emotions
        for pred_idx in predicted_classes.cpu().numpy():
            go_emotion = go_emotions_labels[pred_idx]
            mapped_emotion = goemotions_to_7.get(go_emotion, 'neutral')
            all_predictions.append(mapped_emotion)
    
    return all_predictions

# Run predictions
print("\nPredicting on test set...")
test_texts = df_test['Translation'].tolist()

# Monitor GPU usage
import time
start = time.time()
predictions = predict_emotions_gpu(test_texts, batch_size=64)
print(f"Prediction time: {time.time()-start:.2f} seconds")

# Calculate metrics
accuracy = accuracy_score(df_test['corrected'], predictions)
precision = precision_score(df_test['corrected'], predictions, average='weighted')
recall = recall_score(df_test['corrected'], predictions, average='weighted')
f1 = f1_score(df_test['corrected'], predictions, average='weighted')

print(f"\nResults with RoBERTa-GoEmotions:")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print("\nDetailed classification report:")
print(classification_report(df_test['corrected'], predictions))

CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Model on GPU: True

Predicting on test set...
Prediction time: 1.09 seconds

Results with RoBERTa-GoEmotions:
Accuracy:  0.812
Precision: 0.829
Recall:    0.812
F1-score:  0.801

Detailed classification report:
              precision    recall  f1-score   support

       anger       0.70      0.76      0.73        34
     disgust       0.67      0.50      0.57         4
        fear       1.00      0.08      0.15        25
   happiness       0.77      0.81      0.79       145
     neutral       0.86      0.90      0.88       711
     sadness       0.96      0.36      0.53        66
    surprise       0.43      0.61      0.51        57

    accuracy                           0.81      1042
   macro avg       0.77      0.58      0.59      1042
weighted avg       0.83      0.81      0.80      1042

